# Fig. 4 and Sup. Fig. 10: EnrichMap scores of the breast cancer signatures

Every EnrichMap signature of 09_00_G0_arrest_in_breast_cancer and 09_01_HRD_in_breast_cancer (G0 arrest,
proliferation, 13 hallmarks, HRD with four weightings) scored with the null model (default since 0.3.0), plus the
expression-matched random gene sets behind the procedure-matched nulls of Fig. 4c and 4d. The scores are saved for
REV_Fig4_SupFig10_2_panels.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import enrichmap as em
import matplotlib.pyplot as plt
import pickle

In [3]:
sc.set_figure_params(frameon=False, fontsize=6)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"
os.makedirs(source_dir, exist_ok=True)

# scores read by REV_Fig4_SupFig10_2_panels
scores_dir = working_dir + "data/rev_cache/Fig4_SupFig10/"

In [5]:
from helpers import (
    present_in_data,
    package_fitted_signed_weights,
    gene_mean_sd,
    mean_expression_bins_fig4,
    expression_matched_gene_index,
    expression_matched_updown_sets,
)

In [6]:
# 1. Load data (09_00 / 09_01 cells 5-6) and signatures (09_00 cells 7 and 14, 09_01 cell 8)
adata = sc.read_h5ad(processed_data + "adata_slides_0_3.h5ad")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs["batch"] = adata.obs["batch"].astype(str)
var = pd.Index(adata.var_names)

upregulated_genes = [
    "CFLAR", "CALCOCO1", "YPEL3", "CST3", "SERINC1", "CLIP4", "PCYOX1", "TMEM59","RGS2", "YPEL5", "CD63", "KIAA1109",
    "CDH13", "GSN", "MR1", "CYB5R1", "AZGP1","ZFYVE1", "DMXL1", "EPS8L2", "PTTG1IP", "MIR22HG", "PSAP", "GOLGA8B",
    "NEAT1", "TXNIP", "MTRNR2L12"
]

downregulated_genes = [
    "NCAPD2", "PTBP1", "MPHOSPH9", "NUCKS1", "TCOF1", "SART3", "SNRPA", "KIF22", "HSP90AA1", "WBP11", "CAD", "SF3B2",
    "KHSRP", "WDR76", "NUP188", "HSP90AB1", "HNRNPM", "SMARCB1", "PNN", "RBBP7", "NPRL3", "USP10", "SGTA", "MRPL4",
    "PSMD3", "KPNB1", "CBX1", "LRRC59", "TMEM97", "NSD2", "PRPF19", "PTGES3", "CPSF6", "SRSF3", "TCERG1", "SMC4",
    "EIF4G1", "ZNF142", "MSH6", "MRPL37", "SFPQ", "STMN1", "ARID1A", "PROSER1", "DDX39A", "EXOSC9", "USP22", "DEK",
    "DUT", "ILF3", "DNMT1", "NASP", "HMGB1P5", "SRRM1", "GNL2", "RNF138", "SRSF1", "TRA2B", "SMPD4", "ANP32B", "HMGA1",
    "MDC1", "HADH", "ARHGDIA", "PRCC", "HDGF", "SF3B4", "UBAP2L", "ILF2", "PARP1", "LBR", "CNOT9", "PPRC1", "SSRP1",
    "CCT5", "DLAT", "HNRNPU", "LARP1", "SCAF4", "RRP1B", "RRP1", "CHCHD4", "GMPS", "RFC4", "SLBP", "PSIP1", "HNRNPK",
    "SKA3", "DIS3L", "USP39", "GPS1", "PA2G4", "HCFC1", "SLC19A1", "ETV4", "RAD23A", "DCTPP1", "RCC1", "EWSR1", "ALYREF",
    "PTMA", "HMGB1", "POM121", "MCMBP", "TEAD4", "CHAMP1", "TOP1", "PRRC2A", "RBM14", "HMGB1P6", "POM121C", "UHRF1"
]
proliferation = ["MKI67", "PCNA", "CCND1", "CCNE1", "CCNA1", "CCNB1", "MCM2", "MCM3", "MCM4", "MCM5", "MCM6", "MCM7", "H3F3A", "H3F3B"]

with open(project_dir + "hallmarks_genelists.pkl", "rb") as f:
    cancer_hallmark_genesets = pickle.load(f)

df = pd.read_csv(project_dir + "signature_all_phenotypes.csv", index_col=0)
hrd_gt = {k: v for k, v in df["HRD"].dropna().to_dict().items() if k in adata.var_names}
hrd_pos = {k: v for k, v in hrd_gt.items() if v > 0}
hrd_neg = {k: v for k, v in hrd_gt.items() if v < 0}

In [7]:
# 2. Signed weights: published route (build_signed_weights, 09_00 cell 8 / 09_01 cell 16) and the weights that
# em.tl.score fits for up/down lists (the same rule)
G0_signature = em.tl.build_signed_weights(
    adata, up_genes=upregulated_genes, down_genes=downregulated_genes, score_key="G0"
)
inferred_signed_dict = em.tl.build_signed_weights(
    adata, up_genes=list(hrd_pos.keys()), down_genes=list(hrd_neg.keys()), score_key="hrd_inferred_signed"
)
signed_lists = {
    "G0": (upregulated_genes, downregulated_genes),
    "hrd_inferred_signed": (list(hrd_pos), list(hrd_neg)),
}

weights_package = {k: package_fitted_signed_weights(adata, *lists) for k, lists in signed_lists.items()}
summary_rows = []
for k, w in [("G0", G0_signature["G0"]), ("hrd_inferred_signed", inferred_signed_dict["hrd_inferred_signed"])]:
    up, down = signed_lists[k]
    summary_rows.append({
        "signature": k,
        "genes": len(w),
        "max |w|": max(abs(x) for x in w.values()),
        "listed up genes with negative weight": sum(w[g] < 0 for g in present_in_data(up, var)),
        "listed down genes with positive weight": sum(w[g] > 0 for g in present_in_data(down, var)),
    })
weight_table = pd.DataFrame(summary_rows)

In [8]:
# 3. EnrichMap (null), with the original calls (batch_key="batch", 6-NN smoothing);
# the signed programmes are passed as up/down lists, so the null model refits their weights for every control set
variants = {
    "null": dict(correct_spatial_covariates=False, compute_q_vals=True),
}
labels = {"null": "EnrichMap (null)"}

for name, kwargs in variants.items():
    em.tl.score(adata, gene_set={f"G0_{name}": {"up": upregulated_genes, "down": downregulated_genes}}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_set={f"proliferation_{name}": proliferation}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_set={f"{h}_{name}": g for h, g in cancer_hallmark_genesets.items()}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_weights={f"hrd_gt_{name}": hrd_gt}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_weights={f"hrd_pos_{name}": hrd_pos}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_set={f"hrd_inferred_{name}": list(hrd_pos.keys())}, batch_key="batch", **kwargs)
    em.tl.score(adata, gene_set={f"hrd_inferred_signed_{name}": {"up": list(hrd_pos), "down": list(hrd_neg)}}, batch_key="batch", **kwargs)

signature_names = ["G0", "proliferation", *cancer_hallmark_genesets, "hrd_gt", "hrd_pos", "hrd_inferred", "hrd_inferred_signed"]
scores = {
    labels[name]: pd.DataFrame({s: adata.obs[f"{s}_{name}_score"].values for s in signature_names}, index=adata.obs_names)
    for name in variants
}
null_info = {
    s: pd.DataFrame({k: adata.uns["null_model"][f"{s}_null"][k] for k in ("null_mean", "null_sd", "p_empirical")}, index=adata.obs_names)
    for s in signature_names
}

Scoring hrd_inferred_null: 124/124 genes found: 100%|██████████| 1/1 [00:00<00:00,  6.08it/s]
Scoring hrd_inferred_signed_null: 219/219 genes found: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]


In [9]:
# 4. Published route for the signed programmes (gene_weights = build_signed_weights): the null model then keeps the
# fitted weights for its control sets (inherited, not refitted)
for name, kwargs in variants.items():
    em.tl.score(adata, gene_weights={f"{k}_published_{name}": w for k, w in {**G0_signature, **inferred_signed_dict}.items()}, batch_key="batch", **kwargs)

published_route = {
    labels[name]: pd.DataFrame({s: adata.obs[f"{s}_published_{name}_score"].values for s in signed_lists}, index=adata.obs_names)
    for name in variants
}

summary_rows = []

for s in signature_names:
    z = scores["EnrichMap (null)"][s]
    row = {
        "signature": s,
        "null z mean": z.mean(),
        "null z sd": z.std(),
        "null z max |z|": z.abs().max(),
        "null z > 1.96": (z > 1.96).mean(),
        "null z < -1.96": (z < -1.96).mean(),
        "p_empirical <= 0.05": (null_info[s]["p_empirical"] <= 0.05).mean(),
    }
    if s in signed_lists:
        zi = published_route["EnrichMap (null)"][s]
        row["null z (inherited weights) > 1.96"] = (zi > 1.96).mean()
        row["null z (inherited weights) < -1.96"] = (zi < -1.96).mean()
    summary_rows.append(row)
null_summary = pd.DataFrame(summary_rows)

Scoring hrd_inferred_signed_published_null: 219/219 genes found: 100%|██████████| 2/2 [00:00<00:00, 157.97it/s]


In [10]:
# 5. Null-model diagnostics of every signature (adata.uns["null_model"])
slides = list(adata.obs["batch"].unique())
summary_rows = []

for s in signature_names:
    info = adata.uns["null_model"][f"{s}_null"]
    offset = np.asarray(info["mean_level_offset"])
    summary_rows.append({
        "signature": s,
        "weights refitted": info["weights_refitted"],
        "control overlap (Jaccard)": info["control_overlap"],
        "level offset ratio": info["level_offset_ratio"],
        **{f"mean level offset, slide {b}": offset[i] for i, b in enumerate(slides)},
    })
null_diag = pd.DataFrame(summary_rows)

In [11]:
# 7. Expression-matched random gene sets (25 mean-expression bins)
gene_means, gene_sds = gene_mean_sd(adata.X)
bins = mean_expression_bins_fig4(gene_means)

In [12]:
# 8. Null-P and null-G0 for the Fig. 4c cross-Moran r (200 draws each)
n_null = 200
up_p, dn_p = present_in_data(upregulated_genes, var), present_in_data(downregulated_genes, var)
prolif_p = present_in_data(proliferation, var)
excl = var.get_indexer(present_in_data(upregulated_genes + downregulated_genes + proliferation, var))

# null-P: random proliferation-like 14-gene sets, CV weights inferred as for proliferation
nullP_idx = expression_matched_gene_index(bins, var.get_indexer(prolif_p), n_null, seed=7, exclude=excl)
nullP_sets = {f"nullP_{k:03d}": list(dict.fromkeys(var[nullP_idx[k]])) for k in range(n_null)}

# null-G0: random signed signatures with the G0 up/down sizes, passed as up/down lists like G0 itself
nullG0_sets = expression_matched_updown_sets(up_p, dn_p, n_null, seed=11, exclude=excl, bins=bins, var_names=var)
nullG0_updown = {f"nullG0_{k:03d}": {"up": u, "down": d} for k, (u, d) in enumerate(nullG0_sets)}

# random sets are scored on a light copy (expression, slide and coordinates only): their gene contributions are not kept
light = ad.AnnData(X=adata.X, obs=adata.obs[["batch"]].copy(), var=pd.DataFrame(index=adata.var_names))
light.obsm["spatial"] = np.asarray(adata.obsm["spatial"])

null_scores = {}

for family, sets in [("nullP", nullP_sets), ("nullG0", nullG0_updown)]:
    null_scores[family] = {}
    for name, kwargs in variants.items():
        tmp = light.copy()
        em.tl.score(tmp, gene_set=sets, batch_key="batch", **kwargs, n_control_sets=200)
        null_scores[family][labels[name]] = pd.DataFrame({k: tmp.obs[f"{k}_score"].values for k in sets}, index=adata.obs_names).astype(np.float32)

Scoring nullG0_199: 126/126 genes found: 100%|██████████| 200/200 [00:28<00:00,  6.97it/s]


In [13]:
# 9. Null-H for the Fig. 4d G0 / proliferation vs hallmark correlations
n_null_h = 30
hallmarks = [f"H{i}" for i in range(1, 14)]
nullH_sets = {}

for hi, h in enumerate(hallmarks):
    h_idx = var.get_indexer(present_in_data(cancer_hallmark_genesets[h], var))
    ni = expression_matched_gene_index(bins, h_idx, n_null_h, seed=300 + hi, exclude=np.unique(np.r_[excl, h_idx]))
    for k in range(n_null_h):
        nullH_sets[f"nullH_{h}_{k:02d}"] = list(dict.fromkeys(var[ni[k]]))

null_scores["nullH"] = {}

for name, kwargs in variants.items():
    tmp = light.copy()
    em.tl.score(tmp, gene_set=nullH_sets, batch_key="batch", **kwargs)
    null_scores["nullH"][labels[name]] = pd.DataFrame(
        {k: tmp.obs[f"{k}_score"].values for k in nullH_sets}, index=adata.obs_names
    ).astype(np.float32)

Scoring nullH_H13_29: 341/341 genes found: 100%|██████████| 390/390 [00:45<00:00,  8.64it/s]  


In [14]:
# 11. Save the scores for REV_Fig4_SupFig10_2_panels
os.makedirs(scores_dir, exist_ok=True)
with open(scores_dir + "scores.pkl", "wb") as f:
    pickle.dump({
        "scores": scores,
        "null_info": null_info,
        "null_summary": null_summary,
        "null_diag": null_diag,
        "published_route": published_route,
        "weight_check": weight_table,
        "weights": {"G0": G0_signature["G0"], "hrd_gt": hrd_gt, "hrd_pos": hrd_pos, "hrd_neg": hrd_neg, "hrd_inferred_signed": inferred_signed_dict["hrd_inferred_signed"]},
        "weights_package": weights_package,
    }, f)
with open(scores_dir + "null_scores.pkl", "wb") as f:
    pickle.dump({"scores": null_scores, "nullP_sets": nullP_sets, "nullG0_sets": nullG0_sets, "nullH_sets": nullH_sets}, f)

In [15]:
adata.write_h5ad(processed_data + "adata_slides_0_3_scores.h5ad")